In [1]:
pip show torch transformers

Name: torch
Version: 2.10.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /home/jupyter/.local/lib/python3.10/site-packages
Requires: cuda-bindings, filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvshmem-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, torchvision
---
Name: transformers
Version: 5.1.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transf

## Установка библиотек

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, classification_report

from datasets import load_dataset
import numpy as np
import pandas as pd
import random
from collections import defaultdict
#import matplotlib.pyplot as plt

from transformers import BertConfig, BertModel, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_linear_schedule_with_warmup

In [3]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_random_seed(224)

In [4]:
ds = load_dataset("ai-forever/kinopoisk-sentiment-classification")

Using the latest cached version of the dataset since ai-forever/kinopoisk-sentiment-classification couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/jupyter/datasphere/project/datasetscache/ai-forever___kinopoisk-sentiment-classification/default/0.0.0/4937df51b02a4c748b38bace5d749524fd90ae4a (last modified on Sat Jan 24 11:20:52 2026).


In [5]:
tokenizer = BertTokenizerFast.from_pretrained('DeepPavlov/rubert-base-cased')

In [6]:
def parse_dataset(filepath):
    texts, slots, classes = [], [], []
    with open(filepath, 'r', encoding='utf-8') as file:
        current_text, current_slots, current_classes = [], [], []
        text_id = 0 
        for line in file:
            line = line.strip()
            if line.startswith('# sent_id'):
              sent_text = line.split()[3]
              sent, text = sent_text.split('_')
              if text_id != int(text):
                texts.append(current_text)
                slots.append(current_slots)
                classes.append(current_classes)
                text_id = int(text)
                current_text, current_slots, current_classes = [], [], []
            elif line.startswith('# text'):
              continue
            elif not line:
              continue
            else:
                token = line.split('\t')
                current_text.append(token[1])
                current_classes.append(token[-1])
                current_slots.append(token[-2])
        if current_text:
          texts.append(current_text)
          slots.append(current_slots)
          classes.append(current_classes)
    return texts, slots, classes

In [7]:
from collections import defaultdict

def extract_classes_and_slots(filepath1):
    classes = set()
    slots = set()
    with open(filepath1, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            fields = line.split('\t')
            if len(fields) < 11:
                continue
            class_field = fields[11]
            slot_field = fields[10]
            classes.add(class_field)
            slots.add(slot_field)
    return classes, slots

def classes_slots_dicts(filepath1, filepath2):
    train_sets = extract_classes_and_slots(filepath1)
    val_sets = extract_classes_and_slots(filepath2)
    classes = set(train_sets[0]).union(set(val_sets[0]))
    slots = set(train_sets[1]).union(set(val_sets[1]))
    print(classes)
    print(slots)

    classes = sorted(classes)
    slots = sorted(slots)
    classes.append('PAD')
    slots.append('PAD')

    class2idx = {cls: idx for idx, cls in enumerate(classes)}
    idx2class = {idx: cls for cls, idx in class2idx.items()}

    slot2idx = {slot: idx for idx, slot in enumerate(slots)}
    idx2slot = {idx: slot for slot, idx in slot2idx.items()}

    return {
        'classes': classes,
        'slots': slots,
        'class2idx': class2idx,
        'idx2class': idx2class,
        'slot2idx': slot2idx,
        'idx2slot': idx2slot
    }

In [8]:
class SemDataset(Dataset):
    def __init__(self, texts, slots, classes, labels, tokenizer, slot2id, class2id, max_length):
        self.texts = texts
        self.slots = slots
        self.classes = classes
        self.labels = labels
        self.tokenizer = tokenizer
        self.slot2id = slot2id
        self.class2id = class2id
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        if isinstance(idx, slice):
            return self.slice_token(idx)
        elif isinstance(idx, int):
            return self.get_instance(idx)

    def align_tokens_and_labels(self, tokens, slots, classes):
        word_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        aligned_slots = []
        aligned_classes = []

        current_word_idx = 0
        for word in word_ids:
            current_word = tokens[current_word_idx]
            original_word_token_count = len(self.tokenizer.tokenize(current_word))

            for subtoken in range(original_word_token_count):
                aligned_slots.append(slots[current_word_idx])
                aligned_classes.append(classes[current_word_idx])
            current_word_idx += 1
        return aligned_slots, aligned_classes
    
    def get_instance(self, index):
        tokens = self.texts[index]
        classes = self.classes[index]
        slots = self.slots[index]
        label = self.labels[index]

        encoding = self.tokenizer(
                    tokens,
                    is_split_into_words=True,
                    padding='max_length',
                    truncation=True,
                    max_length=self.max_length,
                    return_tensors='pt'
                )

        slots, classes = self.align_tokens_and_labels(tokens, slots, classes)

        pad_len = self.max_length - len(slots)
        if pad_len > 0:
            for i in range(pad_len):
                slots.append('PAD')
                classes.append('PAD')
        else:
            slots = slots[:self.max_length]
            classes = classes[:self.max_length]

        slots = [self.slot2id[slot] for slot in slots]
        classes = [self.class2id[cls] for cls in classes]

        encoding["slots"] = slots
        encoding["classes"] = classes
        encoding["label"] = label
        encoding["input_ids"] = torch.squeeze(encoding["input_ids"], 0)
        encoding["token_type_ids"] = torch.squeeze(encoding["token_type_ids"], 0)
        encoding["attention_mask"] = torch.squeeze(encoding["attention_mask"], 0)

        return {key: torch.tensor(val) for key, val in encoding.items()}

    def slice_token(self, index):
        start, stop, step = index.indices(len(self.texts))
        result = []
        for index in range(start, stop, step):
            item = self.get_instance(self, index)
            result.append({key: torch.tensor(val) for key, val in item.items()})
        return result

In [9]:
def create_loader(path_to_dataset, labels, tokenizer, slots2id, classes2id, max_length):
    texts, slots, classes = parse_dataset(path_to_dataset)
    dataset = SemDataset(texts, slots, classes, labels, tokenizer, slots2id, classes2id, max_length)
    return DataLoader(dataset, batch_size, shuffle=False)

In [10]:
batch_size = 16
max_length = 512
epochs = 3
lstm_hidden_size=64
semantic_emb_dim=128

In [ ]:
train_dataset_raw = 'sentiment_train_pred.conllu'
val_dataset_raw = 'sentiment_val_pred.conllu'
sem_labels_dict = classes_slots_dicts(train_dataset_raw, val_dataset_raw)

train_labels = ds['train']['label']
val_labels = ds['validation']['label']

In [12]:
train_loader = create_loader(train_dataset_raw, train_labels, tokenizer, sem_labels_dict['slot2idx'], sem_labels_dict['class2idx'], max_length=max_length)
val_loader = create_loader(val_dataset_raw, val_labels, tokenizer, sem_labels_dict['slot2idx'], sem_labels_dict['class2idx'], max_length=max_length)

In [38]:
import warnings
warnings.filterwarnings("ignore")

In [13]:
class BiLSTMPooling(nn.Module):
    def __init__(self, emb_dim, hidden_size):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, token_embeddings):
        # token_embeddings: [batch_size, seq_len, emb_dim]
        lstm_out, _ = self.lstm(token_embeddings)  # [batch_size, seq_len, 2*hidden_size]
        return lstm_out.mean(dim=1)  # Mean pooling over sequence

In [43]:
class SentimentClassifier(nn.Module):
  def __init__(self, num_labels, num_semantic_classes, num_semantic_slots, semantic_emb_dim, lstm_hidden_size):
    super().__init__()
    self.bert = BertModel.from_pretrained("DeepPavlov/rubert-base-cased")
    self.drop = nn.Dropout(p=0.3)

    self.semantic_class_embedding = nn.Embedding(num_semantic_classes, semantic_emb_dim)
    self.semantic_slot_embedding = nn.Embedding(num_semantic_slots, semantic_emb_dim)

    self.semantic_lstm = BiLSTMPooling(emb_dim=semantic_emb_dim, hidden_size=lstm_hidden_size)

    # Классификатор — BERT + BiLSTM(class) + BiLSTM(slot)
    self.classifier = nn.Linear(self.bert.config.hidden_size + 2 * 2 * lstm_hidden_size, num_labels)

  def forward(self, input_ids=None, token_type_ids=None, attention_mask=None, slots=None, classes=None, labels=None):
  #def forward(self, input_ids, attention_mask):
    _, pooled_output = self.bert(
      input_ids=input_ids,
      attention_mask=attention_mask,
      return_dict=False)

    mask = attention_mask.unsqueeze(-1).float()
    # Эмбеддинги семантики: предполагается, что slots и classes — (batch_size,)
    semantic_class_embeds = self.semantic_class_embedding(classes)  # (batch_size, sem_emb_dim)
    semantic_slot_embeds = self.semantic_slot_embedding(slots)      # (batch_size, sem_emb_dim)

    semantic_class_embeds = semantic_class_embeds * mask
    semantic_slot_embeds = semantic_slot_embeds * mask

    class_summary = self.semantic_lstm(semantic_class_embeds)  # (batch_size, 2 * hidden_size)
    slot_summary = self.semantic_lstm(semantic_slot_embeds)    # (batch_size, 2 * hidden_size)

        # Объединяем
    combined = torch.cat([pooled_output, class_summary, slot_summary], dim=1)  # (batch_size, ...)
    
    logits = self.classifier(combined)
    
    if labels is not None:
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(logits.view(-1, self.classifier.out_features), labels.view(-1))
        return {"loss": loss, "logits": logits}
    else:
        return {"logits": logits}
    
  def save_pretrained(self):
    # Сделать все параметры континуальными
    state_dict = self.state_dict()
    for key, value in state_dict.items():
        if not value.is_contiguous():
            state_dict[key] = value.contiguous()
    # Сохранить состояние
    torch.save(state_dict, "./pytorch_model.bin")

In [44]:
num_labels = len(set(ds['train']['label']))
num_semantic_classes = len(sem_labels_dict['classes'])
num_semantic_slots = len(sem_labels_dict['slots'])

model = SentimentClassifier(num_labels=num_labels,
    num_semantic_classes=num_semantic_classes,
    num_semantic_slots=num_semantic_slots,
    semantic_emb_dim=semantic_emb_dim,
    lstm_hidden_size=lstm_hidden_size)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 615.65it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from diff

In [45]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [17]:
def compute_metrics(p):

    predictions, labels = p

    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [46]:
training_args = TrainingArguments(
    output_dir="model/checkpoints",
    eval_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    logging_dir="outputs/logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    #gradient_checkpointing=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [47]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,
    eval_dataset=val_loader.dataset,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    #optimizers=(optimizer, scheduler)
    )

In [48]:
# собственно обучение - автоматически делает логи
trainer.train()

# оценим модельку
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")
# Сохраним, что получилось
#trainer.save_model("./ner_model")
#b5a31e3a762dc4fdbd905c7a205899ee8116917a

  1%|          | 10/1971 [00:15<50:38,  1.55s/it]
                                                  
  1%|          | 10/1971 [00:15<50:38,  1.55s/it]]

{'loss': '1.113', 'grad_norm': '5.491', 'learning_rate': '4.977e-05', 'epoch': '0.01522'}


  1%|          | 20/1971 [00:30<49:34,  1.52s/it]
                                                  
  1%|          | 20/1971 [00:30<49:34,  1.52s/it]]

{'loss': '1.097', 'grad_norm': '4.023', 'learning_rate': '4.952e-05', 'epoch': '0.03044'}


  2%|▏         | 30/1971 [00:46<49:47,  1.54s/it]
                                                  
  2%|▏         | 30/1971 [00:46<49:47,  1.54s/it]]

...

 33%|███▎      | 650/1971 [17:21<35:06,  1.59s/it]
                                                    
 33%|███▎      | 650/1971 [17:21<35:06,  1.59s/it]t]

{'loss': '0.7553', 'grad_norm': '11.32', 'learning_rate': '3.354e-05', 'epoch': '0.9893'}


 33%|███▎      | 657/1971 [17:31<26:20,  1.20s/it]

  0%|          | 0/94 [00:00<?, ?it/s]

  2%|▏         | 2/94 [00:00<00:30,  2.97it/s]

  3%|▎         | 3/94 [00:01<00:43,  2.08it/s]

  4%|▍         | 4/94 [00:02<00:49,  1.83it/s]

  5%|▌         | 5/94 [00:02<00:51,  1.72it/s]

  6%|▋         | 6/94 [00:03<00:54,  1.62it/s]

  7%|▋         | 7/94 [00:04<00:56,  1.55it/s]

  9%|▊         | 8/94 [00:04<00:54,  1.58it/s]

 10%|▉         | 9/94 [00:05<00:53,  1.58it/s]

 11%|█         | 10/94 [00:05<00:53,  1.57it/s]

 12%|█▏        | 11/94 [00:06<00:50,  1.63it/s]

 13%|█▎        | 12/94 [00:07<00:51,  1.61it/s]

 14%|█▍        | 13/94 [00:07<00:49,  1.64it/s]

 15%|█▍        | 14/94 [00:08<00:49,  1.61it/s]

 16%|█▌        | 15/94 [00:08<00:48,  1.62it/s]

 17%|█▋        | 16/94 [00:09<00:47,  1.65it/s]

 18%|█▊        | 17/94 [00:10<00:47,  1.63it/s]

 19%|█▉        | 18/94 [00:10<00:46,  1.62it/s]

 20%|██        | 19/94 [00:11<00:46,  1.62it/s]

 21%|██▏       | 20/94 [00:12<00:4

Confusion Matrix:
 [[303 177  20]
 [ 55 210 235]
 [  4  67 429]]
{'eval_loss': '0.7811', 'eval_accuracy': '0.628', 'eval_f1': '0.6226', 'eval_precision': '0.6423', 'eval_recall': '0.628', 'eval_runtime': '58.92', 'eval_samples_per_second': '25.46', 'eval_steps_per_second': '1.596', 'epoch': '1'}


ValueError: You are trying to save a non contiguous tensor: `bert.encoder.layer.0.attention.self.query.weight` which is not allowed. It either means you are trying to save tensors which are reference of each other in which case it's recommended to save only the full tensors, and reslice at load time, or simply call `.contiguous()` on your tensor to pack it before saving.

In [ ]:
#optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5) #correct_bias=False
total_steps = len(train_dataloader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
  optimizer,
  num_warmup_steps=0,
  num_training_steps=total_steps
)

loss_fn = nn.CrossEntropyLoss().to(device)

In [ ]:
def train_epoch(
  model,
  data_loader,
  loss_fn,
  optimizer,
  device,
  scheduler,
  n_examples
):
  model = model.train()

  losses = []
  all_preds = []
  all_labels = []

  for d in data_loader:
    input_ids = d["input_ids"].to(device)
    attention_mask = d["attention_mask"].to(device)
    targets = d["label"].to(device)
    slots = d['slots'].to(device)
    classes = d['classes'].to(device)

    outputs = model(
      input_ids=input_ids,
      attention_mask=attention_mask,
      slots=slots,
      classes=classes)

    loss = loss_fn(outputs, targets)

    all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
    all_labels.extend(targets.cpu().numpy())

    losses.append(loss.item())

    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

  precision, recall, f1, _ = precision_recall_fscore_support(
  all_labels, all_preds, average='binary', zero_division=0)
  acc = accuracy_score(all_labels, all_preds)
  print(confusion_matrix(all_labels, all_preds))

  return precision, recall, f1, acc, np.mean(losses)

In [ ]:
def eval_model(model, data_loader, loss_fn, device, n_examples):
  model = model.eval()

  losses = []
  all_preds = []
  all_labels = []

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["label"].to(device)
      slots = d['slots'].to(device)
      classes = d['classes'].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        slots=slots,
        classes=classes
      )

      loss = loss_fn(outputs, targets)

      all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
      all_labels.extend(targets.cpu().numpy())

      losses.append(loss.item())

  precision, recall, f1, _ = precision_recall_fscore_support(
  all_labels, all_preds, average='binary', zero_division=0)
  acc = accuracy_score(all_labels, all_preds)
  print(confusion_matrix(all_labels, all_preds))

  return precision, recall, f1, acc, np.mean(losses)

In [ ]:
%%time

history = defaultdict(list)
#best_accuracy = 0

for epoch in range(epochs):

  print(f'Epoch {epoch + 1}/{epochs}')
  print('-' * 10)

  train_precision, train_recall, train_f1, train_acc, train_loss = train_epoch(
    model,
    train_dataloader,
    loss_fn,
    optimizer,
    device,
    scheduler,
    1000
  )

  print(f'Train loss {train_loss} train_acc {train_acc} train_precision {train_precision}, train_recall {train_recall}, train_f1 {train_f1}')

  val_precision, val_recall, val_f1, val_acc, val_loss = eval_model(
    model,
  val_dataloader,
    loss_fn,
    device,
    300
  )

  print(f'Val loss {val_loss} val_acc {val_acc} val_precision {val_precision}, val_recall {val_recall}, val_f1 {val_f1}')
  print()

  history['train_acc'].append(train_acc)
  history['train_loss'].append(train_loss)
  history['val_acc'].append(val_acc)
  history['val_loss'].append(val_loss)
  """
  if val_acc > best_accuracy:
    torch.save(model.state_dict(), 'best_model_state.bin')
    best_accuracy = val_acc
  """

In [ ]:
history['train_acc'] = [score.to('cpu') for score in  history['train_acc']]
history['val_acc'] = [score.to('cpu') for score in  history['val_acc']]

In [ ]:
plt.plot(history['train_acc'], label='train accuracy')
plt.plot(history['val_acc'], label='validation accuracy')

plt.title('Training history')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.ylim([0, 1]);

In [ ]:
test_acc, _ = eval_model(
  model,
  val_dataloader,
  loss_fn,
  device,
  1000
)

test_acc.item()

In [ ]:
def get_predictions(model, data_loader):
  model = model.eval()

  #review_texts = []
  predictions = []
  prediction_probs = []
  real_values = []

  with torch.no_grad():
    for d in data_loader:

      #texts = d["review_text"]
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["label"].to(device)
      slots = d['slots'].to(device)
      classes = d['classes'].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        slots=slots,
        classes=classes
      )
      preds = torch.argmax(outputs, dim=1)

      probs = F.softmax(outputs, dim=1)

      #review_texts.extend(texts)
      predictions.extend(preds)
      prediction_probs.extend(probs)
      real_values.extend(targets)

  predictions = torch.stack(predictions).cpu()
  prediction_probs = torch.stack(prediction_probs).cpu()
  real_values = torch.stack(real_values).cpu()
  return predictions, prediction_probs, real_values

In [ ]:
y_pred, y_pred_probs, y_test = get_predictions(model, val_dataloader)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['0', '1', '2']))

In [ ]:
def show_confusion_matrix(confusion_matrix):
  hmap = sns.heatmap(confusion_matrix, annot=True, fmt="d", cmap="Blues")
  hmap.yaxis.set_ticklabels(hmap.yaxis.get_ticklabels(), rotation=0, ha='right')
  hmap.xaxis.set_ticklabels(hmap.xaxis.get_ticklabels(), rotation=30, ha='right')
  plt.ylabel('True sentiment')
  plt.xlabel('Predicted sentiment');

cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index=['0', '1', '2'], columns=['0', '1', '2'])
show_confusion_matrix(df_cm)



---

старое

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy="epoch",    # Evaluate after each epoch
    learning_rate=1.8562022777790423e-05,             # Learning rate
    per_device_train_batch_size=32, # Batch size for training
    per_device_eval_batch_size=32,  # Batch size for evaluation
    num_train_epochs=3,             # Number of epochs
    weight_decay=0.001928128030703447,              # Strength of weight decay
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=10,               # что это такое?
    save_strategy="epoch",          # Save model after each epoch
    load_best_model_at_end=True,    # Load the best model after training
    metric_for_best_model="f1", # Use F1 score to choose the best model
    seed=224
)

In [ ]:
def compute_metrics(p):

    predictions, labels = p
    # логиты в индексы
    predictions = predictions.argmax(axis=-1)
    cm = confusion_matrix(labels, predictions)

    print("Confusion Matrix:\n", cm)
    # пихнем в метрику и получим результат
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
def hp_space_fn(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3,log=True),
        "weight_decay" : trial.suggest_float("weight_decay", 1e-5, 0.1, log=True) # логарифмический масштаб для lr
        }

def model_init():
    return BertForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels = len(set(ds['train']['label'])))

In [ ]:
trainer = Trainer(
    model=model,
    #model_init = model_init,
    args=training_args,
    train_dataset=little_train_loader.dataset,  # Training dataset
    eval_dataset=little_val_loader.dataset,   # Evaluation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    #optimizers=(optimizer, scheduler)
    )


In [ ]:
!pip install optuna

In [ ]:
best_run = trainer.hyperparameter_search(
    hp_space=hp_space_fn,
    n_trials=5, # количество запусков с разными гиперпараметрами
    direction="maximize", # максимизируем f1
    backend="optuna" # backend для поиска по сетке
)

In [ ]:
best_run.hyperparameters
#{'learning_rate': 1.8562022777790423e-05, 'weight_decay': 0.001928128030703447}

In [ ]:
!wandb login --relogin

In [ ]:
# собственно обучение - автоматически делает логи
trainer.train()

# оценим модельку
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# Сохраним, что получилось
trainer.save_model("./ner_model")
#b5a31e3a762dc4fdbd905c7a205899ee8116917a

In [ ]:
def eval_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      #print(outputs)
      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

In [ ]:
test = eval_model(model, test_loader, device)

In [ ]:
test

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(torch.tensor(ds['test']['label']), torch.zeros(1500), average='macro')
acc = accuracy_score(torch.tensor(ds['test']['label']), torch.zeros(1500))
print({
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall})

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True) # Подключите Google Drive
!cp -r /content/results /content/drive/MyDrive/чекпоинты # Замените на ваши пути